# Entrenamiento CORTEX-ML en Google Colab (por partes)

Este notebook permite entrenar el cerebro Macro de 1,017,856 neuronas LIF por bloques en Colab gratuito, guardando el progreso en tu bóveda privada de Hugging Face después de cada sesión.

## Antes de empezar
1. Consigue tu token de Hugging Face: https://huggingface.co/settings/tokens
2. En la primera celda, pega tu token donde dice `TU_TOKEN_AQUI`.
3. Cambia el entorno de ejecución a **T4 GPU** (Runtime > Change runtime type > T4 GPU).
4. Ejecuta las celdas en orden.

---
⚠️ **Limitaciones de Colab gratuito:** cada sesión tiene tiempo límite y puede cortarse. Guarda siempre antes de cerrar.


In [ ]:
# Celda 1: Configurar token de Hugging Face e instalar dependencias
import os

# PEGA AQUÍ TU TOKEN DE HUGGING FACE (entre comillas)
os.environ['HF_TOKEN'] = 'TU_TOKEN_AQUI'

!pip install -q huggingface_hub
print('Dependencias instaladas.')

In [ ]:
# Celda 2: Descargar archivos necesarios del repositorio
!git clone -b arena/886cbd4f-nuby --single-branch https://github.com/alexsndersoto04-source/nuby-.git /content/nuby_repo 2>/dev/null || true
!cp -r /content/nuby_repo/cortex-lm /content/cortex-lm 2>/dev/null || true
!cp -r /content/nuby_repo/cloud /content/cloud 2>/dev/null || true
!ls -la /content/cortex-lm


In [ ]:
# Celda 3: Compilar el motor C++ de CORTEX-ML
!cd /content/cortex-lm && make portable -j$(nproc)
!ls -la /content/cortex-lm/bin/


In [ ]:
# Celda 4: Descargar el último cerebro desde Hugging Face
!python /content/cloud/colab_sync.py download
!ls -lh /content/cortex-lm/data/cerebro_macro_1m.bin


In [ ]:
# Celda 5: Descargar un bloque de Wikipedia en español para entrenar
!pip install -q datasets
from datasets import load_dataset

# Cargamos un pequeño muestreo de Wikipedia ES (ajusta el tamaño según RAM disponible)
ds = load_dataset('wikimedia/wikipedia', '20231101.es', split='train', streaming=True)

textos = []
chars = 0
max_chars = 200000  # ~200 KB por bloque (ajustable)

for ejemplo in ds:
    t = ejemplo['text']
    textos.append(t)
    chars += len(t)
    if chars >= max_chars:
        break

bloque = '\n'.join(textos)
with open('/content/bloque_wikipedia.txt', 'w', encoding='utf-8') as f:
    f.write(bloque)

print(f'Bloque guardado: {chars} caracteres, {len(textos)} artículos')


In [ ]:
# Celda 6: Entrenar el cerebro con el bloque (alimentar continuo)
!cd /content/cortex-lm && ./bin/cerebro alimentar \
    /content/cortex-lm/data/cerebro_macro_1m.bin \
    /content/bloque_wikipedia.txt \
    --bloque 200000 \
    --epocas 1 \
    --lr 0.24 \
    --ancla 0.022 \
    --semantica --max-pasajes 24

print('Entrenamiento del bloque completado.')


In [ ]:
# Celda 7: Subir el cerebro actualizado a Hugging Face (¡IMPORTANTE antes de cerrar!)
!python /content/cloud/colab_sync.py upload
print('Progreso guardado en Hugging Face. Puedes cerrar la sesión con seguridad.')


---
## Para la siguiente sesión
1. Abre este mismo notebook.
2. Pon tu token en la Celda 1.
3. Ejecuta todas las celdas.
4. La Celda 4 descargará el cerebro actualizado y continuará desde donde lo dejaste.

Cuando la cuota de T4 gratuita se agote, puedes esperar a que se renueve o usar otra cuenta de Google.
